# TabPFN ablation: do commute features help? (Swindon)

Same 137 Swindon LSOAs / 27 MSOAs, target `log_total_GVA_2023`, identical **GroupKFold by MSOA**.
Compares **engineered features only** vs **engineered + commute features** to isolate the
commute contribution.

> Note: this is *within-Swindon* CV. It is **not** comparable to the `caafe_total_gva_features.ipynb`
> result (R²≈0.69), which trains on the rest of the country (Others, 988 rows) and tests on Swindon.
> Commute features only exist for the 27 Swindon MSOAs, so they cannot be added to that
> Others→Swindon transfer setup — only to this within-Swindon evaluation.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

target = "log_total_GVA_2023"
ids = ["LSOA21CD", "MSOA21CD"]

# 1) engineered features (in ../regression-tabpfn+llm/) -> keep Swindon only
base = pd.read_csv("../regression-tabpfn+llm/total_gva_engineered_features.csv")
base = base[base["is_swindon"] == "Swindon"].copy()

# 2) merge MSOA-level commute features (co-located in commuting-regression/)
commute = pd.read_csv("msoa_commute_features_swindon.csv")  # 27 Swindon MSOAs
df = base.merge(commute, on="MSOA21CD", how="inner")

# 3) two feature sets for the ablation (same 137 rows, same target)
eng_feats     = [c for c in base.columns     if c not in ids + ["is_swindon", target]]
commute_feats = [c for c in commute.columns  if c != "MSOA21CD"]

feature_sets = {
    "engineered only":      eng_feats,
    "engineered + commute": eng_feats + commute_feats,
}

y = df[target].values
groups = df["MSOA21CD"].values
print("rows:", df.shape[0], "| MSOAs:", df["MSOA21CD"].nunique())
print("engineered:", len(eng_feats), "| commute:", len(commute_feats))

rows: 137 | MSOAs: 27
engineered: 10 | commute: 9


In [2]:
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error, r2_score,
)

reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)

In [3]:
# Ablation: same 137 Swindon rows, same GroupKFold by MSOA, with vs without commute
n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

rows = []
oof_store = {}
for name, fset in feature_sets.items():
    Xm = df[fset].values
    oof = np.zeros(len(df))
    for tr_idx, te_idx in gkf.split(Xm, y, groups):
        reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
        reg.fit(Xm[tr_idx], y[tr_idx])
        oof[te_idx] = reg.predict(Xm[te_idx])
    oof_store[name] = oof
    rows.append({
        "feature_set": name, "n_feats": len(fset),
        "MAE":  mean_absolute_error(y, oof),
        "RMSE": np.sqrt(mean_squared_error(y, oof)),
        "MAPE": mean_absolute_percentage_error(y, oof),
        "R2":   r2_score(y, oof),
    })

res = pd.DataFrame(rows)
print(f"target: {target}  |  GroupKFold by MSOA ({n_splits} folds), 137 Swindon LSOAs\n")
print(res.to_string(index=False))
print(f"\ndelta R2 (commute - baseline): {res.iloc[1]['R2'] - res.iloc[0]['R2']:+.4f}")

target: log_total_GVA_2023  |  GroupKFold by MSOA (5 folds), 137 Swindon LSOAs

         feature_set  n_feats      MAE     RMSE     MAPE       R2
     engineered only       10 0.471516 0.752270 0.131276 0.659486
engineered + commute       19 0.482252 0.752773 0.135509 0.659031

delta R2 (commute - baseline): -0.0005


In [4]:
# Export out-of-fold predictions (engineered + commute variant) for residuals / IPI / maps
oof = oof_store["engineered + commute"]
out = df[ids].copy()
out["y_true"] = y
out["y_pred_oof"] = oof
out["residual"] = out["y_true"] - out["y_pred_oof"]
out.to_csv("tabpfn_commute_predictions.csv", index=False)
print("saved tabpfn_commute_predictions.csv", out.shape)
out.head()

saved tabpfn_commute_predictions.csv (137, 5)


,LSOA21CD,MSOA21CD,y_true,y_pred_oof,residual
0,E01015471,E02006849,2.776581,2.294603,0.481978
1,E01015473,E02003219,6.207100,6.174655,0.032445
2,E01015475,E02003226,4.612086,4.724710,-0.112623
3,E01015477,E02003226,3.035241,3.326383,-0.291141
4,E01015478,E02003232,5.274071,3.646910,1.627160
